# 模型部署契约与版本元数据

## 学习目标

为模型推理定义稳定的输入校验、预处理、输出格式和版本元数据，并验证保存后的模型可以被独立加载。

## 概念模型

部署模型不只是一个 `.pt` 文件，还包括模型结构、权重、输入 shape、预处理参数、类别映射和训练版本。缺少其中任何一项都可能导致线上结果错误。

In [ ]:
import json
import tempfile
from pathlib import Path
import torch
from torch import nn

class DeployedClassifier(nn.Module):
    def __init__(self, features=4, classes=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(features, 8), nn.ReLU(), nn.Linear(8, classes))
    def forward(self, x): return self.net(x)

model = DeployedClassifier().eval()
artifact_metadata = {'model_version': 'demo-1', 'input_shape': [4], 'class_names': ['negative', 'positive'], 'normalization': {'mean': 0.0, 'std': 1.0}}
print(artifact_metadata)

### 实验 1：输入校验和推理函数

**实验目的**：在模型调用前拒绝错误 ndim、feature shape 和空 batch，并返回类别 id、名称与置信度。越早校验，错误越容易定位且不会进入昂贵计算。

部署契约还应固定 dtype、数值范围、预处理、设备、最大 batch 和类别表版本。softmax 最大值只是模型置信度，不一定经过校准。


In [ ]:
def infer(model, inputs, metadata):
    expected = tuple(metadata['input_shape'])
    if inputs.ndim != 2 or tuple(inputs.shape[1:]) != expected:
        raise ValueError(f'expected (batch, {expected}), got {tuple(inputs.shape)}')
    if inputs.shape[0] == 0:
        raise ValueError('empty batch')
    if not inputs.is_floating_point():
        inputs = inputs.float()
    mean = metadata['normalization']['mean']
    std = max(metadata['normalization']['std'], 1e-6)
    with torch.inference_mode():
        probabilities = model((inputs - mean) / std).softmax(dim=1)
    ids = probabilities.argmax(dim=1)
    names = metadata['class_names']
    return [{'class_id': int(i), 'label': names[int(i)], 'confidence': float(p[int(i)])} for i, p in zip(ids, probabilities)]

result = infer(model, torch.randn(3, 4), artifact_metadata)
print(result)
assert len(result) == 3
try: infer(model, torch.randn(3, 5), artifact_metadata)
except ValueError as error: print('expected validation error:', error)

### 实验 2：保存 artifact 并 round-trip

**实验目的**：分别保存权重和 JSON metadata，再创建同结构模型恢复，验证推理输出一致。artifact 是权重、模型结构、预处理和类别映射的组合，不是单个 `.pt` 文件。

加载时应校验 schema/model version、输入契约和类别数，并使用 `map_location`。生产发布还需要原子写入、校验和与回滚策略。


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    torch.save(model.state_dict(), directory / 'weights.pt')
    (directory / 'metadata.json').write_text(json.dumps(artifact_metadata), encoding='utf-8')
    restored = DeployedClassifier().eval()
    restored.load_state_dict(torch.load(directory / 'weights.pt', weights_only=True))
    restored_metadata = json.loads((directory / 'metadata.json').read_text())
    test_input = torch.randn(2, 4)
    with torch.inference_mode():
        torch.testing.assert_close(model(test_input), restored(test_input))
    assert restored_metadata['model_version'] == 'demo-1'
    print('artifact round-trip passed')

## 官方教程补充

**对应官方源文件：** `intermediate_source/torch_export_tutorial.py`、`beginner_source/onnx/intro_onnx.py`、`recipes_source/recipes/saving_and_loading_models_for_inference.rst`

官方部署材料要求把模型之外的契约显式化：输入名称/shape/dtype、动态维约束、预处理、输出语义、类别表和版本。导出只是转换步骤，必须在目标 runtime 上做数值一致性与边界输入测试。模型 artifact 与任意 Python pickle 的信任边界不同；加载外部权重时优先使用受限模式并验证来源。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

列出部署 artifact 必须包含的元数据；说明为什么只加载权重而不加载预处理参数会造成输入分布错误。

## 试一试

增加输入范围校验和模型版本校验；把类别映射改成三类并验证输出接口仍然稳定。

## 常见错误与调试

部署时忘记 `eval()`、预处理参数与训练不一致、类别顺序变化、只保存权重不保存结构版本、没有测试错误输入和空 batch。